# ChlamyCodonTransformer Fine-tuning

C. reinhardtii 葉緑体コドン最適化のための CodonTransformer ファインチューニング

**実行前チェックリスト:**
1. ランタイム → ランタイムのタイプを変更 → **GPU (A100推奨, T4でも可)**
2. `training_data.json` を Google Drive の `MyDrive/ChlamyDesign/data/` にアップロード済み
3. （任意）HuggingFace token を取得 (write権限): https://huggingface.co/settings/tokens

**学習完了後:**
- `models/ChlamyCodonTransformer/` (HuggingFace 形式) をローカルにコピー
- または HuggingFace Hub にアップロード (PUSH_TO_HUB=True)

## 0. Google Drive マウント & パス設定

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR     = '/content/drive/MyDrive/ChlamyDesign'
TRAINING_DATA = f'{DRIVE_DIR}/data/training_data.json'
MODEL_OUTPUT  = f'{DRIVE_DIR}/models/ChlamyCodonTransformer'
LOG_DIR       = f'{DRIVE_DIR}/logs'

os.makedirs(f'{DRIVE_DIR}/models', exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

assert os.path.exists(TRAINING_DATA), (
    f'training_data.json が見つかりません:\n  {TRAINING_DATA}\n'
    f'Google Drive の MyDrive/ChlamyDesign/data/ にアップロードしてください。'
)

import json
with open(TRAINING_DATA) as f:
    lines = f.readlines()
N_SAMPLES = len(lines)
print(f'✓ 訓練データ: {TRAINING_DATA}')
print(f'✓ サンプル数: {N_SAMPLES:,}')
print(f'✓ モデル出力: {MODEL_OUTPUT}')

## 1. パッケージインストール & GPU確認

In [ ]:
!pip install -q 'CodonTransformer>=1.0.0' 'transformers>=4.30.0' 'accelerate>=0.20.0'

import torch
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA使用可 : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM     : {vram:.1f} GB')
else:
    print('⚠ GPU が検出されません。ランタイムの設定を確認してください。')

## 2. ハイパーパラメータ設定

In [ ]:
import math

# ── ハイパーパラメータ (CodonTransformer 論文準拠) ──
BATCH_SIZE   = 6
MAX_EPOCHS   = 10
LR           = 5e-5
WARMUP_RATIO = 0.1

STEPS_PER_EPOCH = math.ceil(N_SAMPLES / BATCH_SIZE)
TOTAL_STEPS     = STEPS_PER_EPOCH * MAX_EPOCHS
WARMUP_STEPS    = int(TOTAL_STEPS * WARMUP_RATIO)

print(f'サンプル数      : {N_SAMPLES}')
print(f'バッチサイズ    : {BATCH_SIZE}')
print(f'1エポックのstep : {STEPS_PER_EPOCH}')
print(f'合計 steps      : {TOTAL_STEPS}  ({MAX_EPOCHS} epochs)')
print(f'warmup steps    : {WARMUP_STEPS}')

## 3. モデル & トークナイザー読み込み

In [ ]:
from transformers import BigBirdForMaskedLM
from CodonTransformer.CodonPrediction import load_tokenizer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'デバイス: {device}')

print('ベースモデルを HuggingFace Hub から読み込み中...')
model     = BigBirdForMaskedLM.from_pretrained('adibvafa/CodonTransformer')
tokenizer = load_tokenizer()

n_params = sum(p.numel() for p in model.parameters())
print(f'パラメータ数: {n_params:,}')
print('✓ 読み込み完了')

## 4. データセット & コレーター準備

In [ ]:
import json
from torch.utils.data import Dataset
from transformers import DataCollatorForLanguageModeling

class CodonDataset(Dataset):
    """Map-style dataset for CodonTransformer fine-tuning.
    
    IterableJSONData (CodonTransformer 同梱) は iterator プロパティが
    未実装のため使用不可。代わりに Map-style dataset を実装。
    """
    def __init__(self, data_path, tokenizer, max_length=2048):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.samples = []
        with open(data_path) as f:
            for line in f:
                rec = json.loads(line)
                self.samples.append(f"{rec['organism']} {rec['codons']}")
        print(f'  読み込み: {len(self.samples)} サンプル')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.samples[idx],
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt',
        )
        return {k: v.squeeze(0) for k, v in enc.items()}

train_dataset = CodonDataset(TRAINING_DATA, tokenizer)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15,
)

print('✓ データセット準備完了')

## 5. HuggingFace Hub ログイン（任意）

In [ ]:
# HuggingFace にアップロードする場合のみ設定
PUSH_TO_HUB  = False               # True にするとアップロード
HF_USERNAME  = 'your_hf_username'  # ← 自分のユーザー名に変更
HF_REPO_ID   = f'{HF_USERNAME}/ChlamyCodonTransformer'

if PUSH_TO_HUB:
    from huggingface_hub import login
    login()  # トークン入力プロンプトが表示されます
    print(f'✓ HuggingFace Hub ログイン完了: {HF_REPO_ID}')
else:
    print('HuggingFace アップロード: 無効 (PUSH_TO_HUB=True で有効化)')

## 6. ファインチューニング実行

In [ ]:
from transformers import Trainer, TrainingArguments
import time

training_args = TrainingArguments(
    output_dir=MODEL_OUTPUT,
    num_train_epochs=MAX_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    learning_rate=LR,
    warmup_steps=WARMUP_STEPS,
    weight_decay=0.01,
    # ログ & 保存
    logging_dir=LOG_DIR,
    logging_steps=50,
    save_strategy='epoch',
    save_total_limit=3,
    # HuggingFace Hub
    push_to_hub=PUSH_TO_HUB,
    hub_model_id=HF_REPO_ID if PUSH_TO_HUB else None,
    # 高速化
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
)

print(f'ファインチューニング開始')
print(f'  lr={LR}, batch={BATCH_SIZE}, epochs={MAX_EPOCHS}, warmup={WARMUP_STEPS} steps')

t0 = time.time()
trainer.train()
elapsed = time.time() - t0
print(f'\n✓ 学習完了: {elapsed/60:.1f} 分')

## 7. モデル保存

In [ ]:
# HuggingFace 形式で保存 (config.json + model.safetensors + tokenizer)
# model_manager.py の from_pretrained() で直接読み込み可能
trainer.save_model(MODEL_OUTPUT)
tokenizer.save_pretrained(MODEL_OUTPUT)
print(f'✓ HF形式で保存: {MODEL_OUTPUT}')

# サブディレクトリにもバックアップ
final_path = f'{MODEL_OUTPUT}/final'
os.makedirs(final_path, exist_ok=True)
trainer.save_model(final_path)
tokenizer.save_pretrained(final_path)
print(f'✓ バックアップ : {final_path}')

if PUSH_TO_HUB:
    trainer.push_to_hub(commit_message='ChlamyCodonTransformer: fine-tuned on C.reinhardtii chloroplast CDS')
    print(f'✓ HuggingFace Hub: https://huggingface.co/{HF_REPO_ID}')

print()
print('━' * 50)
print('次のステップ:')
print(f'  1. Google Drive から {MODEL_OUTPUT}/ をダウンロード')
print(f'  2. ローカルの models/ChlamyCodonTransformer/ に配置')
print(f'  3. API が自動的にモデルをロードします')
print('━' * 50)

## 8. 動作確認 — rbcL タンパク質でテスト

In [ ]:
from CodonTransformer.CodonPrediction import predict_dna_sequence

# rbcL (C. reinhardtii 葉緑体 RuBisCO 大サブユニット)
test_protein = 'MSPQTETKASVGFKAGVKEYKLTYYTPEYETKDTDILAAFRVTPQPGVPPEEAGAAVAAESSTGTWTTVWTDGLTSLDRYKGRCYRIEEGQTINEDTAYFFDQFKQGCDINIFQYDSAGLLNKFEEMDHYREPEHSPESAGGIHVWHMPALTEIFGDDSVLQFGGGTLGHPWGNAPGATN_'

model.eval()
result = predict_dna_sequence(
    protein=test_protein,
    organism='Chlamydomonas reinhardtii chloroplast',
    device=device,
    model=model,
    tokenizer=tokenizer,
)

dna = result.predicted_dna
gc  = sum(1 for b in dna if b in 'GC') / len(dna) * 100

print(f'予測DNA (先頭90 nt): {dna[:90]}')
print(f'GC%: {gc:.1f}%  (目標: ~34%, ベースCT: ~61%)')
print(f'長さ: {len(dna)} nt')

# ベースモデルとの比較のために再読み込みして比較
base_model = BigBirdForMaskedLM.from_pretrained('adibvafa/CodonTransformer').to(device)
base_result = predict_dna_sequence(
    protein=test_protein,
    organism='Chlamydomonas reinhardtii chloroplast',
    device=device, model=base_model, tokenizer=tokenizer,
)
base_gc = sum(1 for b in base_result.predicted_dna if b in 'GC') / len(base_result.predicted_dna) * 100

print(f'\n--- 比較 ---')
print(f'ChlamyCT  GC%: {gc:.1f}%')
print(f'Base CT   GC%: {base_gc:.1f}%')
print(f'改善量       : {base_gc - gc:+.1f}pp')